# Connect to RabbitMQ

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from communication import protocol
from communication.rabbitmq import Rabbitmq

# Initialize RabbitMQ connection (adjust parameters as needed)
try:
    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    print("✓ Connected to RabbitMQ successfully")
except Exception as e:
    print(f"✗ Failed to connect to RabbitMQ: {e}")
    print("\nMake sure RabbitMQ is running. You can start it with:")
    print("  python -m startup.start_docker_rabbitmq")

def send_control_message(rmq, msg):
    """Send a control message to the UR3e Mockup via RabbitMQ."""
    try:
        rmq.send_message(
            routing_key=protocol.ROUTING_KEY_CTRL,
            message=msg
        )
        print(f"✓ Control message: {msg} sent successfully")
    except Exception as e:
        print(f"✗ Failed to send control message: {e}")

✓ Connected to RabbitMQ successfully


# Automatic movement generator

Launch the code to perform random movement, it makes new movement each time it detects that the robot has stopped
N.B : You have to connect to run the code above before

In [6]:
import time
import numpy as np
from influxdb_client import InfluxDBClient

UPDATE_INTERVAL = 0.5     # Time between velocity checks
TIMEOUT_LIMIT = 15        
THRESHOLD = 0.01          # Criteria for stationary state
URL = "http://localhost:8086"
TOKEN = "SECRET_AF"
ORG = "ur3e"
BUCKET = "ur3e"
client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
query_api = client.query_api()

def get_max_robot_velocity(api, bkt):
    """Get the maximum joint velocity from the last 3 seconds."""
    query = f'''
    from(bucket: "{bkt}")
      |> range(start: -3s)
      |> filter(fn: (r) => r["_measurement"] == "sensor_data")
      |> filter(fn: (r) => r["_field"] =~ /qd_actual_joint_[0-5]/)
      |> last()
    '''
    try:
        result = api.query(query)
        velocities = [abs(record.get_value()) for table in result for record in table.records]
        return max(velocities) if velocities else None
    except Exception:
        return None

print("Starting Random Data Generation...")

try:
    while True:
       
        target_position = (np.random.uniform(-1, 1, 6) * np.pi).tolist() # Joint positions within [-pi, pi]
        
        msg_load = {
            protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
            protocol.CtrlMsgKeys.JOINT_POSITIONS: [target_position],
            protocol.CtrlMsgKeys.MAX_VELOCITY: 60,
            protocol.CtrlMsgKeys.ACCELERATION: 80,
        }
        
        send_control_message(rmq, msg_load)
        send_control_message(rmq, { protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY })

        time.sleep(1.5) # Initial delay to allow the robot to start moving
        is_moving = True
        start_time = time.time()
        
        while is_moving:
            max_vel = get_max_robot_velocity(query_api, BUCKET)
        
            # Exit condition: Robot is stationary OR Safety timeout reached
            if (max_vel is not None and max_vel < THRESHOLD) or (time.time() - start_time > TIMEOUT_LIMIT):
                current_time = time.strftime('%H:%M:%S')
                print(f"[{current_time}] Target reached (Max Vel: {max_vel if max_vel else 'N/A'}). Proceeding...")
                is_moving = False
            else:
                time.sleep(UPDATE_INTERVAL)

except KeyboardInterrupt:
    print("\nGeneration stopped")

Starting Random Data Generation...
✓ Control message: {'type': 'load_program', 'joint_positions': [[2.9596984102090698, -1.4587543184943634, 2.69192529254577, 2.2071181772418855, -3.0289082114915944, -0.5004588238688094]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully
[09:11:48] Target reached (Max Vel: N/A). Proceeding...
✓ Control message: {'type': 'load_program', 'joint_positions': [[0.32762856232433313, -0.34290914368609066, -2.9220730527112, 1.0284794109180775, -2.6453468109734994, -1.2183445796014414]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully
[09:11:55] Target reached (Max Vel: N/A). Proceeding...
✓ Control message: {'type': 'load_program', 'joint_positions': [[0.5362635289440177, -1.6744945424327573, -0.28116521512169945, -0.3666362509283925, 0.6128141624047526, -1.3844926413625793]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓

# Random movement generator

The block below will create random movements. Every time the arm has finished a movement press ctrl+enter to generate a new movement.

In [ ]:
import numpy as np
import time

# Construct control message for loading a program
position_np = (np.random.rand((6)) - 0.5) * 0.5*np.pi * 1000 # Generate 6 random joint positions
position_np = np.ones((6))*0*np.pi
position = position_np.tolist()
vel = 60 # deg/s
acc = 1 # deg/s²

msg = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
    protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
    protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
    protocol.CtrlMsgKeys.ACCELERATION: acc,
}

send_control_message(rmq, msg)

# send control message for starting program
msg_start = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
}

send_control_message(rmq, msg_start)


✓ Control message: {'type': 'load_program', 'joint_positions': [[0, 0, 0, 0, 0, 0]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully


# Automatic movement generator

In [ ]:
import time
import numpy as np

while True:
    # Construct control message for loading a program
    position_np = (np.random.normal(0, 1, (6)) - 0.5) * 2*np.pi # Generate 6 random joint positions [-pi, pi]
    position = position_np.tolist()

    max_vel_bounds = (0, 360) # deg/s
    max_acc_bounds = (0, 180) # deg/s²      *Assumed

    def random_from_range(range: tuple):
        return np.random.rand() * (range[1] - range[0]) + range[0]

    vel = random_from_range(max_vel_bounds) # deg/s
    acc = random_from_range(max_acc_bounds) # deg/s²

    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
        protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
        protocol.CtrlMsgKeys.ACCELERATION: acc,
    }

    send_control_message(rmq, msg)

    # send control message for starting program
    msg_start = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    }

    send_control_message(rmq, msg_start)

    time.sleep(10)

    

# TODO:
# - Should generate random movement CTRL messages
# - The movements should be uniformly distributed
# - A new message should only be sent after the previous motion is done (Wait for stationarity)
  # - Lowpass filter over past N states, if velocity approx 0, send new control message
# - The messages should contain random joint positions, max velocity and acceleration


✓ Control message: {'type': 'load_program', 'joint_positions': [[-19.58924495623814, -17.538457706849304, -11.278325899943237, 5.377998093878092, -19.43913942367269, 2.297211596937503]], 'max_velocity': 141.48220400460127, 'acceleration': 32.58822446278252} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[4.9780617800155165, -6.265175630765215, 3.375417345868476, 2.3004014889422875, 1.7679712964092593, -11.06515902671603]], 'max_velocity': 155.50408841133932, 'acceleration': 87.78685300526705} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[-3.3504912526626884, -0.7746226608229099, -4.932318238395204, 0.5498340430926866, -3.718618653723524, -24.0598717610822]], 'max_velocity': 316.7597359373423, 'acceleration': 58.54641201433692} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control m

KeyboardInterrupt: 